# Set up

In [ ]:
import numpy as np
import torch
import gpytorch
import matplotlib.pyplot as plt


from scipy.integrate import odeint
import pandas as pd
from math import pi

import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import functools
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import trange

import tqdm as tqdm


from scipy.integrate import odeint
from scipy.signal import find_peaks

from sklearn.preprocessing import StandardScaler

from typing import Dict, Any, Callable, Tuple

import linear_operator

# Data

In [ ]:
train_x = np.load("Data/LV_train_x.npy")    # 形状 (M, 2)
train_y = np.load("Data/LV_train_y.npy")    # 形状 (M, 2*K)

In [ ]:
train_x = torch.tensor(train_x, dtype=torch.float32)

train_y = torch.tensor(train_y, dtype=torch.float32)


# MGP

In [ ]:
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(MultitaskGPModel, self).__init__(train_x, train_y, likelihood)

        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.ZeroMean(), num_tasks=train_y.shape[1]
        )
        self.covar_module = gpytorch.kernels.MultitaskKernel(
            gpytorch.kernels.RBFKernel(), num_tasks=train_y.shape[1], rank=1
        )


    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

In [ ]:
def train_MultitaskGP(X_train, Y_train, lr=0.05, num_iterations=5000, patience=10, device='cpu', disable_progbar=True):

    X_train = X_train.to(device)
    Y_train = Y_train.to(device)


    likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=Y_train.shape[1])
    model = MultitaskGPModel(X_train, Y_train, likelihood)

    model = model.to(device)
    likelihood = likelihood.to(device)

    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    
    best_loss = float('inf')
    counter = 0
    iterator = tqdm.tqdm(range(num_iterations), disable=disable_progbar)

    for i in iterator:
        optimizer.zero_grad()
        output = model(X_train)
        loss = -mll(output, Y_train)
        loss.backward()
        if not disable_progbar:
            iterator.set_postfix(loss=loss.item())
        optimizer.step()

        if loss <= best_loss:
            best_loss = loss
            best_state = model.state_dict()  
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                model.load_state_dict(best_state)  
                break

    return model, likelihood

In [ ]:
gp_model, gp_likelihood = train_MultitaskGP(train_x, train_y, lr=0.05, num_iterations=5000, patience=10, device='cuda', disable_progbar=False)

In [ ]:
torch.save({
    'model_state_dict': gp_model.state_dict(),
    'likelihood_state_dict': gp_likelihood.state_dict(),
}, "multitask_gp_lv.pth")

In [ ]:
device = 'cpu'

num_tasks = train_y.shape[1]   # 保证和训练时一致
gp_likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
gp_model = MultitaskGPModel(train_x, train_y, gp_likelihood)

checkpoint = torch.load("multitask_gp_lv.pth", map_location=device)
gp_model.load_state_dict(checkpoint['model_state_dict'])
gp_likelihood.load_state_dict(checkpoint['likelihood_state_dict'])

gp_model.eval(); gp_likelihood.eval()


# Dim

In [ ]:
train_x.shape

In [ ]:
dev, dt = train_x.device, train_x.dtype

# Tools

这部分代码不一定用的上

In [ ]:
def _clear_all_caches(module: torch.nn.Module):
    """清理 gpytorch 的 memoize 缓存，避免 dtype/device 切换后出现旧缓存。"""
    try:
        gpytorch.utils.memoize.clear_cache(module)
    except Exception:
        pass
    for m in module.modules():
        try:
            gpytorch.utils.memoize.clear_cache(m)
        except Exception:
            pass


def _to_numpy(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def _dev_dt_from(model, fallback_device="cpu"):
    """从模型抓取 device/dtype；若失败则回退。"""
    try:
        p = next(model.parameters())
        return p.device, p.dtype
    except StopIteration:
        return torch.device(fallback_device), torch.float32
    

def to_dense_safe(op: torch.Tensor) -> torch.Tensor:
    """兼容 linear_operator 和老 LazyTensor 的安全稠密化。"""
    if isinstance(op, torch.Tensor):
        return op
    if hasattr(op, "to_dense"):
        return op.to_dense()
    if hasattr(op, "evaluate"):
        return op.evaluate()
    raise TypeError(f"Object of type {type(op)} is not convertible to dense.")

@torch.no_grad()
def linop_solve(op, rhs: torch.Tensor, jitter: float = 1e-6) -> torch.Tensor:
    """
    统一 (近似) K^{-1} @ rhs 的求解：
      1) 优先使用 LinearOperator.solve(rhs)（新版本）
      2) 其次使用 LazyTensor.inv_matmul(rhs)（旧版本）
      3) 兜底：转稠密 + Cholesky（仅小规模、前两者都不可用时）
    支持 rhs 为 [n] 或 [n, k]。
    """
    squeeze_back = False
    if rhs.dim() == 1:
        rhs = rhs.unsqueeze(-1)
        squeeze_back = True

    if hasattr(op, "solve"):
        sol = op.solve(rhs)
    elif hasattr(op, "inv_matmul"):
        sol = op.inv_matmul(rhs)
    else:
        K = to_dense_safe(op)  # 兜底：稠密求解
        n = K.size(-1)
        eye = torch.eye(n, device=K.device, dtype=K.dtype)
        Kj = K + jitter * eye
        L = torch.linalg.cholesky(Kj)
        # torch.cholesky_solve 在新版本中等价于：
        sol = torch.cholesky_solve(rhs, L)

    if squeeze_back:
        sol = sol.squeeze(-1)
    return sol

@torch.no_grad()
def symmetric_psd_sqrt(A, eps=1e-12):
    """对称(半)正定矩阵 A 的平方根：A^{1/2}（用 eigh，避免复数不稳定）。"""
    w, V = torch.linalg.eigh(A)
    w = torch.clamp(w, min=eps)
    return (V * torch.sqrt(w)) @ V.transpose(-1, -2)

# B, K_xx, K_ff, K_JJ

In [ ]:
@torch.no_grad()

def precompute_structs_lazy(
    model,
    likelihood,
    X: torch.Tensor,
    Y: torch.Tensor,
    dev: torch.device = None,
    dt: torch.dtype = None,
    *,
    jitter: float = 1e-6,
) -> Dict[str, Any]:
    """
    预计算（基于 GPyTorch 的 Lazy/LinearOperator）:
      - B ∈ R^{P×P}           任务核
      - K_xx ∈ R^{N×N}        数据核（RBF）[注意：此处返回 dense；大规模可自行改为 lazy]
      - Σ_noise ∈ R^{P×P}     任务噪声（对角）
      - solve_Kff(v_task)     计算 (B⊗K_xx + I_N⊗Σ)^(-1) @ v_task，输入/输出均为“任务优先”vec
      - alpha_y ∈ R^{PN×1}    K_ff^{-1} vec(Y)（任务优先）
      - K_JJ ∈ R^{(PD)×(PD)}  B ⊗ Kdd（RBF 的常量二阶项；无外层 ScaleKernel 时无 σ^2）
      - 以及 N, D, P, ell 等
    """
    # --- 设备/精度 ---
    if dev is None or dt is None:
        # 从模型参数推断默认 device/dtype
        p = next(model.parameters())
        dev = dev or p.device
        dt  = dt  or p.dtype

    model.to(dev, dtype=dt)
    likelihood.to(dev, dtype=dt)
    model.eval(); likelihood.eval()

    X = X.to(dev, dt)
    Y = Y.to(dev, dt)

    N, D = X.shape
    P    = Y.shape[1]

    # --- 数据核 K_xx（RBF） & lengthscale ---
    # 你的模型结构：MultitaskKernel(RBFKernel(), num_tasks=P, rank=...)
    rbf = model.covar_module.data_covar_module  # == RBFKernel()
    K_xx_lazy = rbf(X)                          # Lazy [N×N]
    K_xx = to_dense_safe(K_xx_lazy)             # 若规模大，建议改为返回 lazy

    ell = rbf.lengthscale.detach().to(dev, dt)  # [1,1,D] 或 [D]

    # --- 任务核 B ---
    B = to_dense_safe(model.covar_module.task_covar_module.covar_matrix)  # [P×P]

    # --- 噪声 Σ_noise（每任务一条对角噪声） ---

    diag_noise = likelihood.task_noises.to(dev, dt)  # [P]
    Sigma_noise = torch.diag(diag_noise)


    inter_idx = torch.arange(N * P, device=dev)
    inter_to_task = inter_idx.view(N, P).transpose(0, 1).reshape(-1)  # interleaved -> task-first
    task_to_inter = torch.empty_like(inter_to_task)
    task_to_inter[inter_to_task] = inter_idx                           # task-first -> interleaved

    def _to_interleaved(v_task: torch.Tensor) -> torch.Tensor:
        if v_task.dim() == 1:
            return v_task[task_to_inter]
        else:
            return v_task[task_to_inter, :]

    def _to_taskfirst(v_inter: torch.Tensor) -> torch.Tensor:
        if v_inter.dim() == 1:
            return v_inter[inter_to_task]
        else:
            return v_inter[inter_to_task, :]



    with gpytorch.settings.fast_pred_var():
        _ = likelihood(model(X))

    alpha_inter = model.prediction_strategy.mean_cache  # [NP]

    alpha_y_task = _to_taskfirst(alpha_inter).reshape(-1, 1)  # [PN, 1]

    # --- K_JJ = B ⊗ Kdd（RBF 的常量二阶项）---
    # 无外层 ScaleKernel 时，k'' 在 x=x' 处给出 diag(1/ell^2)（若有 outputscale 可乘 σ^2）
    if ell.numel() == 1:
        invl2 = 1.0 / (ell.squeeze() ** 2)
        Kdd = torch.eye(D, device=dev, dtype=dt) * invl2
    else:
        invl2 = 1.0 / (ell.view(-1) ** 2)    # [D]
        Kdd = torch.diag(invl2).to(dev, dt)  # [D×D]
    K_JJ = torch.kron(B, Kdd)                # [P*D × P*D]




    return {
        "B": B,                              # [P×P] (dense)
        "K_xx": K_xx,                        # [N×N] (dense；大规模可改为返回 K_xx_lazy)
        "Sigma_noise": Sigma_noise,          # [P×P] (dense，对角)
        # "solve_Kff": solve_Kff,              # Callable：任务优先 vec -> 解（任务优先）
        "alpha_y": alpha_y_task,             # [PN × 1] 任务优先
        "K_JJ": K_JJ,                        # [P*D × P*D]
        "K_dd": Kdd,
        "N": N, "D": D, "P": P, "ell": ell, "dev": dev, "dt": dt,
        # 便于调试/深度使用：
        # "K_train_lazy": K_train_lazy,        # Lazy (NP×NP)
        "perm_task_to_inter": task_to_inter, # 任务优先 -> interleaved
        "perm_inter_to_task": inter_to_task, # interleaved -> 任务优先
    }


# K_Xx

In [ ]:
def rbf_k_xX_grad(model, x, X, dev=None, dt=None):

    if dev is None or dt is None:
        d0, t0 = _dev_dt_from(model)
        dev = dev or d0
        dt  = dt or t0

    x_flat = x.squeeze(0).detach().requires_grad_(True)    # [D]

    def g(x_flat_):
        x1 = x_flat_.unsqueeze(0)                          # [1, D]
        k_row = to_dense_safe(model.covar_module.data_covar_module(x1, X)).squeeze(0)  # [N]
        return k_row

    try:
        J = torch.autograd.functional.jacobian(g, x_flat, vectorize=True)  # [N, D] ; shape: (P, n*, N)
    except TypeError:
        J = torch.autograd.functional.jacobian(g, x_flat)                   # [N, D] ; shape: (P, n*, N)
    dk_dx = J

    return dk_dx.to(dev, dt)

In [ ]:
def rbf_k_xX_and_grad_expre(model, x, X, dev=None, dt=None):
    """
    返回：
      k(x, X)   : (N,)
      dk/dx     : (N×D)
      Kdd       : (D×D) = ∂²k/∂x∂x'|_{x'=x}，RBF 下为常量 -diag(1/ell^2)
    """
    if dev is None or dt is None:
        d0, t0 = _dev_dt_from(model)
        dev = dev or d0
        dt  = dt or t0

    rbf = model.covar_module.data_covar_module
    rbf.to(dev, dtype=dt)
    _clear_all_caches(rbf)

    x   = x.to(dev, dt)               # 1×D
    X   = X.to(dev, dt)               # N×D
    D   = X.shape[1]
    ell = rbf.lengthscale.detach().to(dev, dt).squeeze()

    diff = (x - X).unsqueeze(0)       # 1×N×D

    if ell.ndim == 0:
        invl2 = 1.0/(ell**2)                          # 标量
        sq = (diff**2).sum(-1) * invl2                # 1×N
        k = torch.exp(-0.5*sq).squeeze(0)             # N
        dk_dx = -(diff.squeeze(0) * invl2) * k.unsqueeze(1)  # N×D
        Kdd = torch.eye(D, device=dev, dtype=dt) * invl2
    else:
        invl2 = 1.0/(ell**2)                          # (D,)
        sq = (diff**2 * invl2).sum(-1)                # 1×N
        k = torch.exp(-0.5*sq).squeeze(0)             # N
        dk_dx = -(diff.squeeze(0) * invl2) * k.unsqueeze(1)  # N×D
        Kdd = torch.diag(invl2).to(dev, dt)
    return k.to(dev, dt), dk_dx.to(dev, dt), Kdd.to(dev, dt)

# E(g), mu_J, ...

后面关于方差的sigma不一定用

In [ ]:
@torch.no_grad()
def expected_metric_at_x(model, x, X, structs):
    """
    在单点 x 处计算：
      - E_g = μ_J^T μ_J ∈ R^{D×D}
      - μ_J ∈ R^{P×D}
      - Sigma_J_post = cov(vec(J) | f) ∈ R^{(PD)×(PD)} （忽略观测噪声）
      - Sigma_trace = Σ_p Cov(J_p) 的和 ∈ R^{D×D}

    这里使用近似公式：cov(J | f) = K_JJ - K_Jf K_ff^{-1} K_fJ，
    其中 K_ff ≈ B ⊗ K_xx，不再加噪声项。
    """

    dev, dt = structs["dev"], structs["dt"]
    B, K_xx = structs["B"], structs["K_xx"]
    alpha_y, K_JJ = structs["alpha_y"], structs["K_JJ"]
    N, D, P = structs["N"], structs["D"], structs["P"]

    # 一阶导：dk/dx(x, X) ∈ R^{N×D}
    dk_dx = rbf_k_xX_grad(model, x, X, dev, dt)   # N×D
    dk_dx_T = dk_dx.transpose(0, 1).contiguous()  # D×N

    # K_Jf: Jacobian 与 function values 的先验协方差 (PD×PN)
    K_Jf = torch.kron(B.contiguous(), dk_dx_T)    # (PD)×(PN)

    # 后验均值：μ_J = E[J(x) | Y] ≈ K_Jf (K_ff + Σ)^(-1) y
    mu_vec = K_Jf @ alpha_y                       # (PD)×1
    mu_J   = mu_vec.reshape(P, D)                 # P×D
    E_g    = (mu_J.transpose(0,1) @ mu_J)         # D×D

    # cov(J | f) ≈ K_JJ - K_Jf K_ff^{-1} K_fJ
    K_fJ = K_Jf.transpose(0, 1).contiguous()      # (PN)×(PD)

    K_xx_op = linear_operator.operators.DenseLinearOperator(K_xx)
    B_op    = linear_operator.operators.DenseLinearOperator(B)
    K_ff_op = linear_operator.operators.KroneckerProductLinearOperator(B_op, K_xx_op)

    sol = linop_solve(K_ff_op, K_fJ)             # ≈ K_ff^{-1} K_fJ   (PN×PD)
    middle = K_Jf @ sol                          # (PD×PD)
    Sigma_J_post = K_JJ - middle                 # (PD×PD)

    # Sigma_trace = Σ_p Cov(J_p)
    Sigma_trace = torch.zeros(D, D, device=dev, dtype=dt)
    for p in range(P):
        blk = Sigma_J_post[p*D:(p+1)*D, p*D:(p+1)*D]  # Cov(J_p) ∈ R^{D×D}
        Sigma_trace += blk

    return E_g, mu_J, Sigma_trace, Sigma_J_post